# Atlas Trade — Unsloth Qwen3.5-27B Setup

**Runtime:** Google Colab Pro — A100 80GB  
**Model:** `unsloth/Qwen3.5-27B` (BF16, no quantization)  
**Context:** 16384 tokens

Run cells top to bottom on a fresh A100 runtime. On subsequent sessions, re-run from **Step 4** onwards (model is cached in HF cache after the first download).

## Configuration

Set your GitHub repo URL and Hugging Face token here, or export them as environment variables before running:

```python
import os
os.environ["PROJECT_REPO_URL"] = "https://github.com/<your-username>/unsloth-qwen35-trading-assistant.git"
os.environ["HF_TOKEN"] = "hf_..."
```

In [3]:
import os

REPO_URL = os.environ.get(
    "PROJECT_REPO_URL",
    "https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git",
)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

MODEL_NAME       = "unsloth/Qwen3.5-27B"
MAX_SEQ_LENGTH   = 16384
DTYPE            = None   # None = auto-detect (BF16 on A100)
LOAD_IN_4BIT     = False  # Full BF16 — requires A100 80GB

REPO_DIR = "/content/unsloth-qwen35-trading-assistant"

print(f"Model   : {MODEL_NAME}")
print(f"Context : {MAX_SEQ_LENGTH} tokens")
print(f"4-bit   : {LOAD_IN_4BIT}")
print(f"Repo    : {REPO_URL}")
print(f"HF token: {'set' if HF_TOKEN else 'NOT SET (public models only)'}")

Model   : unsloth/Qwen3.5-27B
Context : 16384 tokens
4-bit   : False
Repo    : https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git
HF token: NOT SET (public models only)


## Step 1 — Clone or update the repository

In [4]:
import subprocess, pathlib, sys

def _run(cmd: list[str], **kwargs) -> str:
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{output}")
    return output

def git_pull_latest():
    """Clone the repo on first run; fast-forward pull on subsequent runs."""
    repo = pathlib.Path(REPO_DIR)
    if repo.exists():
        print(_run(["git", "-C", str(repo), "pull", "--ff-only"]))
    else:
        print(_run(["git", "clone", REPO_URL, str(repo)]))
    # Add repo to Python path so `src` is importable
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    print(f"Repo ready at {repo}")

git_pull_latest()

Cloning into '/content/unsloth-qwen35-trading-assistant'...
Repo ready at /content/unsloth-qwen35-trading-assistant


## Step 2 — Install Unsloth

Uses the official runtime-aware auto-installer — automatically matches the active CUDA/Torch stack on this Colab runtime.

In [ ]:
import subprocess, sys, io, contextlib

# Step 1: Run the auto-installer script to get the correct pip command for this runtime
print("Detecting runtime and generating install command...")
dl = subprocess.run(
    ["wget", "-qO-",
     "https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py"],
    capture_output=True, text=True,
)
if dl.returncode != 0:
    raise RuntimeError("Could not fetch Unsloth installer:\n" + dl.stderr)

# Capture what the script prints (it prints the pip command to run)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    exec(dl.stdout)
pip_cmd = buf.getvalue().strip()

if not pip_cmd:
    # Fallback: plain pip install
    pip_cmd = f"{sys.executable} -m pip install unsloth"

print(f"Install command:\n  {pip_cmd}\n")

# Step 2: Actually run the command
result = subprocess.run(pip_cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("Unsloth installation failed.")

print("\n✅ Unsloth installed.")
print("\n⚠️  RESTART THE RUNTIME NOW, then re-run from Step 1 (skip this cell).")
print("   Runtime menu → Restart session  (or Ctrl+M .)  ")

## Step 3 — Hugging Face login

Required only if you plan to push LoRA adapters or access gated models. Skip if `HF_TOKEN` is empty.

In [6]:
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN set — skipping login (public model download will still work).")

No HF_TOKEN set — skipping login (public model download will still work).


## Step 4 — Load the model

First run downloads ~54 GB of BF16 weights. Subsequent runs load from the HF cache (~1-2 minutes).

In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

FastLanguageModel.for_inference(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

ModuleNotFoundError: No module named 'unsloth'

## Step 5 — Chat function

`chat()` keeps a conversation history so Atlas Trade remembers the context within a session.  
Call `chat(reset=True)` to start a fresh conversation.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages

_conversation_history: list[dict[str, str]] = []

def chat(
    user_message: str,
    market_context: str = "",
    reset: bool = False,
    thinking: bool = False,
    max_new_tokens: int = 2048,
    temperature: float = 0.6,
    top_p: float = 0.95,
) -> str:
    """
    Send a message to Atlas Trade.

    Parameters
    ----------
    user_message    : Your message or analysis request.
    market_context  : Optional structured market data (price, indicators, etc.).
    reset           : Set True to clear conversation history and start fresh.
    thinking        : Set True to enable Qwen3 chain-of-thought thinking mode.
    max_new_tokens  : Maximum tokens to generate.
    temperature     : Sampling temperature (0.6 is a good default for trading analysis).
    top_p           : Nucleus sampling threshold.
    """
    global _conversation_history

    if reset:
        _conversation_history = []
        print("[conversation reset]")

    # Append thinking mode trigger to message if requested
    message = user_message
    if thinking:
        message = message + " /think"

    messages = build_trading_messages(
        user_message=message,
        market_context=market_context,
        history=_conversation_history,
    )

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
    )

    new_tokens = outputs[0][inputs.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Update history (store without the /think suffix for cleaner history)
    _conversation_history.append({"role": "user",      "content": message})
    _conversation_history.append({"role": "assistant", "content": response})

    return response


def clear_history():
    """Alias for chat(reset=True) when you don't want to send a message."""
    global _conversation_history
    _conversation_history = []
    print("[conversation history cleared]")


print("chat() ready. Usage:")
print('  response = chat("BTC 4H analiz et")')
print('  response = chat("stop nereye koymalıyım?")  # devam sorusu')
print('  response = chat("yeni koin", reset=True)    # geçmişi temizle')

## Step 6 — Start trading

Atlas Trade hazır. Aşağıdaki hücreyi düzenleyerek analizini başlat.

In [ ]:
response = chat(
    user_message="Merhaba, BTC/USDT 4H chart'ta trend durumu nasıl görünüyor?",
    market_context="""
Sembol  : BTC/USDT
Timeframe: 4H
Fiyat   : 83,500 USDT
RSI(14) : 54
MACD    : histogram pozitif, sinyal çizgisinin üzerinde
EMA20   : 82,100 (fiyat üzerinde)
EMA50   : 79,800 (fiyat üzerinde)
Hacim   : ortalama üzerinde son 3 mumda
""",
)

print(response)

In [ ]:
# Devam sorusu — geçmiş otomatik taşınır
response = chat("Bu durumda long için en temiz entry bölgesi neresi?")
print(response)

---

## Utility — Update repo without reloading the model

In [ ]:
# Run this cell whenever you push prompt or code changes from VS Code.
# The model stays loaded — only the Python source is refreshed.
git_pull_latest()

# Reload the trading_prompt module to pick up any changes
import importlib, src.trading_prompt as _tp
importlib.reload(_tp)
from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages
print("Prompt module reloaded.")